# Ejercicio 4: Laboratorio de Estrategias de Chunking

Este notebook experimenta con diferentes configuraciones de chunking usando `RecursiveCharacterTextSplitter` de LangChain.

## Paso 1: Preparar el documento de ejemplo

In [7]:
# Instalar dependencias si es necesario
# !pip install langchain langchain-text-splitters

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Documento de ejemplo: artículo sobre inteligencia artificial
documento = """
# Introducción a la Inteligencia Artificial

La inteligencia artificial (IA) es una rama de la informática que busca crear sistemas
capaces de realizar tareas que normalmente requieren inteligencia humana. Estas tareas
incluyen el aprendizaje, el razonamiento, la percepción y la comprensión del lenguaje natural.

## Historia de la IA

El término "inteligencia artificial" fue acuñado por John McCarthy en 1956 durante la
conferencia de Dartmouth. Sin embargo, las ideas sobre máquinas pensantes se remontan a
mucho antes. Alan Turing, en 1950, propuso el famoso Test de Turing como criterio para
determinar si una máquina puede exhibir comportamiento inteligente indistinguible del humano.

Durante las décadas de 1960 y 1970, la IA experimentó un período de optimismo conocido
como la "edad de oro". Los investigadores creían que la IA general estaba a pocas décadas
de distancia. Sin embargo, las limitaciones computacionales y teóricas llevaron a los
"inviernos de la IA", períodos de reducción de financiación e interés.

## Aprendizaje Automático

El aprendizaje automático (machine learning) es un subcampo de la IA que se centra en
desarrollar algoritmos que permiten a las computadoras aprender de los datos sin ser
programadas explícitamente. Los tres paradigmas principales son:

1. Aprendizaje supervisado: el modelo aprende de ejemplos etiquetados.
2. Aprendizaje no supervisado: el modelo descubre patrones en datos sin etiquetar.
3. Aprendizaje por refuerzo: el modelo aprende mediante prueba y error con recompensas.

## Deep Learning

El deep learning o aprendizaje profundo utiliza redes neuronales con múltiples capas
(de ahí "profundo") para aprender representaciones jerárquicas de los datos. Las
arquitecturas más importantes incluyen:

- Redes Neuronales Convolucionales (CNN): especializadas en procesamiento de imágenes.
- Redes Neuronales Recurrentes (RNN): diseñadas para secuencias temporales.
- Transformers: la arquitectura dominante actual para procesamiento de lenguaje natural,
  introducida en el paper "Attention is All You Need" (2017).

## Modelos de Lenguaje

Los modelos de lenguaje grandes (LLMs) como GPT-4, Claude y Llama representan el estado
del arte en procesamiento de lenguaje natural. Estos modelos se entrenan con cantidades
masivas de texto y pueden generar texto coherente, traducir idiomas, resumir documentos
y responder preguntas.

La técnica de RAG (Retrieval-Augmented Generation) complementa estos modelos permitiéndoles
acceder a información externa actualizada, reduciendo las alucinaciones y proporcionando
respuestas más precisas y verificables.
"""

print(f"Longitud total del documento: {len(documento)} caracteres")
print(f"Número de líneas: {len(documento.splitlines())}")

c:\python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Longitud total del documento: 2609 caracteres
Número de líneas: 50


## Paso 2: Experimentar con diferentes configuraciones

In [9]:
configuraciones = [
    {"chunk_size": 100, "chunk_overlap": 0,  "nombre": "Muy pequeño, sin overlap"},
    {"chunk_size": 100, "chunk_overlap": 20, "nombre": "Muy pequeño, con overlap"},
    {"chunk_size": 300, "chunk_overlap": 0,  "nombre": "Mediano, sin overlap"},
    {"chunk_size": 300, "chunk_overlap": 50, "nombre": "Mediano, con overlap"},
    {"chunk_size": 500, "chunk_overlap": 50, "nombre": "Grande, con overlap pequeño"},
    {"chunk_size": 500, "chunk_overlap": 100,"nombre": "Grande, con overlap grande"},
    {"chunk_size": 1000,"chunk_overlap": 200,"nombre": "Muy grande, con overlap"},
]

# Almacenamos resultados para la tabla
resultados = []

for config in configuraciones:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = splitter.split_text(documento)
    
    resultado = {
        "config": f"{config['chunk_size']}, overlap {config['chunk_overlap']}",
        "n_chunks": len(chunks),
        "tam_promedio": sum(len(c) for c in chunks) / len(chunks),
        "tam_min": min(len(c) for c in chunks),
        "tam_max": max(len(c) for c in chunks),
        "total_chars": sum(len(c) for c in chunks),
        "chunks": chunks
    }
    resultados.append(resultado)

    print(f"\n{'='*70}")
    print(f"Configuración: {config['nombre']}")
    print(f"  chunk_size={config['chunk_size']}, chunk_overlap={config['chunk_overlap']}")
    print(f"  Número de chunks: {len(chunks)}")
    print(f"  Tamaño promedio: {resultado['tam_promedio']:.0f} caracteres")
    print(f"  Tamaño mínimo: {resultado['tam_min']} caracteres")
    print(f"  Tamaño máximo: {resultado['tam_max']} caracteres")
    print(f"  Caracteres totales (con overlap): {resultado['total_chars']}")

    # Mostrar los primeros 3 chunks
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n  --- Chunk {i+1} ({len(chunk)} chars) ---")
        # Mostrar solo las primeras y últimas líneas
        lines = chunk.strip().split('\n')
        if len(lines) <= 4:
            for line in lines:
                print(f"  {line}")
        else:
            print(f"  {lines[0]}")
            print(f"  {lines[1]}")
            print(f"  ...")
            print(f"  {lines[-1]}")


Configuración: Muy pequeño, sin overlap
  chunk_size=100, chunk_overlap=0
  Número de chunks: 36
  Tamaño promedio: 71 caracteres
  Tamaño mínimo: 16 caracteres
  Tamaño máximo: 94 caracteres
  Caracteres totales (con overlap): 2557

  --- Chunk 1 (43 chars) ---
  # Introducción a la Inteligencia Artificial

  --- Chunk 2 (86 chars) ---
  La inteligencia artificial (IA) es una rama de la informática que busca crear sistemas

  --- Chunk 3 (86 chars) ---
  capaces de realizar tareas que normalmente requieren inteligencia humana. Estas tareas

Configuración: Muy pequeño, con overlap
  chunk_size=100, chunk_overlap=20
  Número de chunks: 36
  Tamaño promedio: 71 caracteres
  Tamaño mínimo: 16 caracteres
  Tamaño máximo: 94 caracteres
  Caracteres totales (con overlap): 2557

  --- Chunk 1 (43 chars) ---
  # Introducción a la Inteligencia Artificial

  --- Chunk 2 (86 chars) ---
  La inteligencia artificial (IA) es una rama de la informática que busca crear sistemas

  --- Chunk 3 (86 cha

## Paso 3: Análisis detallado del overlap

In [10]:
# Analizar qué información se comparte entre chunks consecutivos
print("\n" + "="*70)
print("ANÁLISIS DE OVERLAP")
print("="*70)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_text(documento)

for i in range(min(len(chunks) - 1, 5)):  # Mostrar máximo 5 transiciones
    chunk_actual = chunks[i]
    chunk_siguiente = chunks[i + 1]

    # Encontrar el texto solapado
    overlap_text = ""
    for length in range(min(len(chunk_actual), len(chunk_siguiente)), 0, -1):
        if chunk_actual.endswith(chunk_siguiente[:length]):
            overlap_text = chunk_siguiente[:length]
            break

    print(f"\nEntre Chunk {i+1} y Chunk {i+2}:")
    if len(overlap_text) > 80:
        print(f"  Overlap encontrado ({len(overlap_text)} chars): \"{overlap_text[:80]}...\"")
    else:
        print(f"  Overlap encontrado ({len(overlap_text)} chars): \"{overlap_text}\"")
    print(f"  Final chunk {i+1}: \"...{chunk_actual[-50:]}\"")
    print(f"  Inicio chunk {i+2}: \"{chunk_siguiente[:50]}...\"")


ANÁLISIS DE OVERLAP

Entre Chunk 1 y Chunk 2:
  Overlap encontrado (0 chars): ""
  Final chunk 1: "...# Introducción a la Inteligencia Artificial"
  Inicio chunk 2: "La inteligencia artificial (IA) es una rama de la ..."

Entre Chunk 2 y Chunk 3:
  Overlap encontrado (0 chars): ""
  Final chunk 2: "...ensión del lenguaje natural.

## Historia de la IA"
  Inicio chunk 3: "El término "inteligencia artificial" fue acuñado p..."

Entre Chunk 3 y Chunk 4:
  Overlap encontrado (0 chars): ""
  Final chunk 3: "...ropuso el famoso Test de Turing como criterio para"
  Inicio chunk 4: "determinar si una máquina puede exhibir comportami..."

Entre Chunk 4 y Chunk 5:
  Overlap encontrado (0 chars): ""
  Final chunk 4: "...portamiento inteligente indistinguible del humano."
  Inicio chunk 5: "Durante las décadas de 1960 y 1970, la IA experime..."

Entre Chunk 5 y Chunk 6:
  Overlap encontrado (0 chars): ""
  Final chunk 5: "...taciones computacionales y teóricas llevaron a los"
  Inicio chunk 6: ""

## Paso 4: Tabla resumen de análisis comparativo

In [11]:
print("\n" + "="*90)
print("TABLA RESUMEN DE RESULTADOS")
print("="*90)
print(f"{'Configuración':<25} | {'N.Chunks':>9} | {'Tam.Prom':>9} | {'Tam.Min':>8} | {'Tam.Max':>8} | {'Total':>8}")
print("-"*90)

for r in resultados:
    print(f"{r['config']:<25} | {r['n_chunks']:>9} | {r['tam_promedio']:>8.0f} | {r['tam_min']:>8} | {r['tam_max']:>8} | {r['total_chars']:>8}")


TABLA RESUMEN DE RESULTADOS
Configuración             |  N.Chunks |  Tam.Prom |  Tam.Min |  Tam.Max |    Total
------------------------------------------------------------------------------------------
100, overlap 0            |        36 |       71 |       16 |       94 |     2557
100, overlap 20           |        36 |       71 |       16 |       94 |     2557
300, overlap 0            |        14 |      184 |       22 |      290 |     2582
300, overlap 50           |        14 |      186 |       22 |      290 |     2600
500, overlap 50           |         8 |      333 |      220 |      466 |     2666
500, overlap 100          |         8 |      333 |      220 |      466 |     2666
1000, overlap 200         |         4 |      655 |      220 |      858 |     2619


## Paso 5: Análisis de corte de ideas

In [12]:
# Analizar si los chunks cortan ideas a mitad
print("\n" + "="*70)
print("ANÁLISIS DE CORTES DE IDEAS")
print("="*70)

# Configuración más problemática: chunks muy pequeños sin overlap
print("\n--- Configuración: 100 chars, sin overlap (más problemática) ---")
splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_small = splitter_small.split_text(documento)

# Contar chunks que terminan a mitad de frase (sin punto final)
cortes_abruptos = 0
for i, chunk in enumerate(chunks_small):
    chunk_limpio = chunk.strip()
    if not chunk_limpio.endswith(('.', ':', '\n', '!', '?')):
        cortes_abruptos += 1
        if cortes_abruptos <= 5:  # Mostrar solo los primeros 5
            print(f"\nChunk {i+1} termina abruptamente:")
            print(f"  '...{chunk_limpio[-60:]}'")

print(f"\nTotal de chunks con corte abrupto: {cortes_abruptos} de {len(chunks_small)} ({100*cortes_abruptos/len(chunks_small):.1f}%)")

# Configuración óptima: chunks medianos con overlap
print("\n--- Configuración: 500 chars, overlap 100 (más equilibrada) ---")
splitter_medium = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_medium = splitter_medium.split_text(documento)

cortes_abruptos_med = 0
for i, chunk in enumerate(chunks_medium):
    chunk_limpio = chunk.strip()
    if not chunk_limpio.endswith(('.', ':', '\n', '!', '?')):
        cortes_abruptos_med += 1

print(f"Total de chunks con corte abrupto: {cortes_abruptos_med} de {len(chunks_medium)} ({100*cortes_abruptos_med/len(chunks_medium):.1f}%)")


ANÁLISIS DE CORTES DE IDEAS

--- Configuración: 100 chars, sin overlap (más problemática) ---

Chunk 1 termina abruptamente:
  '...# Introducción a la Inteligencia Artificial'

Chunk 2 termina abruptamente:
  '... (IA) es una rama de la informática que busca crear sistemas'

Chunk 3 termina abruptamente:
  '... que normalmente requieren inteligencia humana. Estas tareas'

Chunk 5 termina abruptamente:
  '...## Historia de la IA'

Chunk 6 termina abruptamente:
  '...artificial" fue acuñado por John McCarthy en 1956 durante la'

Total de chunks con corte abrupto: 23 de 36 (63.9%)

--- Configuración: 500 chars, overlap 100 (más equilibrada) ---
Total de chunks con corte abrupto: 3 de 8 (37.5%)


## Conclusiones y respuestas a las preguntas de reflexión

### 1. ¿Con qué configuración se cortan más frases a mitad de una idea?
Con **chunk_size=100 y overlap=0**. Un tamaño tan pequeño no permite que quepan párrafos o ideas completas, y sin overlap no hay forma de recuperar el contexto perdido.

### 2. ¿Cuál es el trade-off entre chunk_size pequeño y grande?
- **Pequeño**: Mayor precisión en recuperación pero pérdida de contexto
- **Grande**: Preserva contexto pero puede incluir información irrelevante

### 3. ¿Por qué usa jerarquía de separadores?
Para cortar en los puntos más naturales primero (párrafos > líneas > frases > palabras). Si solo usara un separador, los cortes serían más arbitrarios.

### 4. ¿Cambiarías los separadores para código Python?
Sí, usaría: `["\nclass ", "\ndef ", "\n\n", "\n", " ", ""]`

### 5. ¿Configuración para documento legal vs FAQ?
- **Legal**: chunk_size=800-1000, overlap=150-200 (párrafos densos que necesitan contexto)
- **FAQ**: chunk_size=200-300, overlap=0-50 (cada par pregunta-respuesta es independiente)